# IR Image Colorization & Enhancement (ISRO BAH Hackathon)

This notebook provides the environment to train the ESRGAN and Pix2Pix models on a free Colab GPU.

## 1. Mount Google Drive
Make sure you have uploaded the project folder to your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this path to where you uploaded the project in your Drive
import os
PROJECT_DIR = '/content/drive/MyDrive/ISRO_BAH'
os.chdir(PROJECT_DIR)
!pwd

## 2. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 3. Prepare Dataset
This extracts 128x128 patches from the Landsat GeoTIFFs in the `archive/` folder.

In [ ]:
!python data/prepare_dataset.py --archive_dir archive --output_dir prepared_data

## 4. Train Super-Resolution (ESRGAN)
Trains the model to upscale the thermal band (B10). This does 50 epochs of L1 pretraining and 50 epochs of GAN finetuning.

In [ ]:
!python train_sr.py --data_dir prepared_data --checkpoint_dir checkpoints/sr --epochs_pretrain 50 --epochs_gan 50 --batch_size 16

## 5. Train Colorization (Pix2Pix)
Trains the U-Net + PatchGAN to map IR bands to RGB. Uses semantic constraints (NDVI/NDWI/NDBI).

In [ ]:
!python train_colorize.py --data_dir prepared_data --checkpoint_dir checkpoints/colorize --epochs 200 --batch_size 8

## 6. Evaluation
Calculates PSNR, SSIM, and FID on the test set.

In [ ]:
!python evaluate.py --data_dir prepared_data/test --checkpoint checkpoints/colorize/colorize_best.pth --output_dir outputs/evaluation